In [2]:
import sys
from pathlib import Path

root = Path.cwd()

if root.name in ["train", "notebooks"]:
    root = root.parent

if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import torch
import torch.nn as nn
import pandas as pd

from src.data.dataset import create_dataloaders
from src.models.transformer import TransformerLanguageModel
from src.training.train import fit, evaluate

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

Device: cuda


Cấu hình thí nghiệm

In [4]:
CONTEXT_LENGTH = 64
BATCH_SIZE = 16

EPOCHS = 50
PATIENCE = 5

MODEL_NAME = "transformer"
EXPERIMENT_NAME = f"word_c{CONTEXT_LENGTH}"

torch.manual_seed(42)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

Nạp WikiText-2

In [5]:
train_loader, valid_loader, test_loader, vocab_size = (
    create_dataloaders(
        context_length=CONTEXT_LENGTH,
        batch_size=BATCH_SIZE
    )
)

print("Vocabulary size:", vocab_size)
print("Train batches:", len(train_loader))
print("Valid batches:", len(valid_loader))
print("Test batches:", len(test_loader))

Vocabulary size: 10000
Train batches: 2084
Valid batches: 218
Test batches: 246


Khởi tạo Transformer

In [4]:
model = TransformerLanguageModel(
    vocab_size=vocab_size,
    embedding_dim=256,
    num_heads=4,
    num_layers=2,
    ffn_dim=1024,
    dropout=0.2,
    max_seq_len=256
).to(device)

total_params = sum(
    p.numel()
    for p in model.parameters()
)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

print("Total parameters:", f"{total_params:,}")

Total parameters: 6,775,568


Kiểm tra Forward Pass

In [5]:
X, Y = next(iter(train_loader))

X = X.to(device)

model.eval()

with torch.no_grad():
    logits = model(X)

print("Input shape:", X.shape)
print("Output shape:", logits.shape)
print("Target shape:", Y.shape)

assert logits.shape == (
    X.shape[0],
    X.shape[1],
    vocab_size
)

print("Transformer Forward Pass: OK")

Input shape: torch.Size([16, 64])
Output shape: torch.Size([16, 64, 10000])
Target shape: torch.Size([16, 64])
Transformer Forward Pass: OK


Kiểm tra Causal Mask

In [6]:
model.eval()

x1 = torch.randint(
    0, vocab_size,
    (1, 64),
    device=device
)

x2 = x1.clone()

# Thay đổi token vị trí 20
x2[0, 20] = (x2[0, 20] + 1) % vocab_size

with torch.no_grad():
    out1 = model(x1)
    out2 = model(x2)

# Các output ở vị trí 0 đến 19 không được thay đổi
assert torch.allclose(
    out1[:, :20, :],
    out2[:, :20, :],
    atol=1e-5,
    rtol=1e-5
)

print("Causal Mask: OK")

Causal Mask: OK


Huấn luyện Transformer

In [8]:
result = fit(
    model=model,
    train_loader=train_loader,
    valid_loader=valid_loader,
    optimizer=optimizer,
    criterion=criterion,
    device=device,
    model_name=MODEL_NAME,
    experiment_name=EXPERIMENT_NAME,
    context_length=CONTEXT_LENGTH,
    batch_size=BATCH_SIZE,
    root=root,
    epochs=10,
    patience=PATIENCE
)


Epoch 1/10


Training:   0%|          | 0/2084 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/218 [00:00<?, ?it/s]

Train Loss: 5.2246 | Train PPL: 185.79
Valid Loss: 4.7703 | Valid PPL: 117.95
Đã lưu best model!

Epoch 2/10


Training:   0%|          | 0/2084 [00:00<?, ?it/s]

...

Evaluating:   0%|          | 0/218 [00:00<?, ?it/s]

Train Loss: 3.9942 | Train PPL: 54.28
Valid Loss: 4.2973 | Valid PPL: 73.50
Đã lưu best model!

Epoch 10/10


Training:   0%|          | 0/2084 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/218 [00:00<?, ?it/s]

Train Loss: 3.9480 | Train PPL: 51.83
Valid Loss: 4.2812 | Valid PPL: 72.33
Đã lưu best model!

Training time: 6.34 phút


Đánh giá Test Set

In [9]:
model.load_state_dict(
    torch.load(
        result["checkpoint_path"],
        map_location=device,
        weights_only=True
    )
)

test_loss, test_ppl = evaluate(
    model,
    test_loader,
    criterion,
    device
)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test PPL: {test_ppl:.2f}")

Evaluating:   0%|          | 0/246 [00:00<?, ?it/s]

Test Loss: 4.2265
Test PPL: 68.48


In [10]:
summary = pd.DataFrame([{
    "model": MODEL_NAME,
    "experiment": EXPERIMENT_NAME,
    "parameters": total_params,
    "context_length": CONTEXT_LENGTH,
    "batch_size": BATCH_SIZE,
    "best_valid_loss": result["best_valid_loss"],
    "best_valid_ppl": result["best_valid_ppl"],
    "test_loss": test_loss,
    "test_ppl": test_ppl,
    "epochs_trained": result["epochs_trained"],
    "training_time_seconds": result["training_time_seconds"]
}])

summary.to_csv(
    Path(result["results_dir"]) / "summary.csv",
    index=False
)

display(summary)

,model,experiment,parameters,context_length,batch_size,best_valid_loss,best_valid_ppl,test_loss,test_ppl,epochs_trained,training_time_seconds
0,transformer,word_c64,6775568,64,16,4.281231,72.329444,4.226522,68.478644,10,380.366612


In [11]:
import json

from src.training.inference import (
    predict_next_token,
    generate_text
)

with open(
    root / "data" / "processed" / "word2idx.json",
    encoding="utf-8"
) as f:
    word2idx = json.load(f)

idx2word = {
    idx: word
    for word, idx in word2idx.items()
}

model.load_state_dict(
    torch.load(
        result["checkpoint_path"],
        map_location=device,
        weights_only=True
    )
)

model.eval()

print("Đã nạp model!")

Đã nạp model!


In [12]:
prompt = "The president of the"

predictions = predict_next_token(
    model=model,
    text=prompt,
    word2idx=word2idx,
    idx2word=idx2word,
    device=device,
    context_length=CONTEXT_LENGTH,
    top_k=5
)

print("Input:", prompt)
print("\nTop 5 predictions:")

for item in predictions:
    print(
        f"{item['token']:15s} "
        f"{item['probability'] * 100:.2f}%"
    )

Input: The president of the

Top 5 predictions:
Lords           4.37%
Goat            3.78%
National        2.30%
FIA             1.97%
United          1.82%


In [13]:
prompt = "The history of the"

generated = generate_text(
    model=model,
    prompt=prompt,
    word2idx=word2idx,
    idx2word=idx2word,
    device=device,
    context_length=CONTEXT_LENGTH,
    max_new_tokens=100,
    temperature=0.8,
    top_k=20
)

print("Prompt:", prompt)
print("\nGenerated text:")
print(generated)

Prompt: The history of the

Generated text:
The history of the country ' s population was largely fluid . The population was also the largest amount of $ 8 million per year at the time of $ 100 million in damages .
